# SepsisShield — Person 2: Data Cleaning and Preprocessing

**Purpose:** Audit data integrity, make only justified structural corrections, document outlier flags and prepare a consistent dataset for Person 3.

**Input:** `data/Dataset.csv`.

**Output:** `outputs/02_preprocessing/cleaned_dataset.csv.gz` plus audit reports.

> This dataset contains intentionally unrecorded clinical measurements. Missing values are preserved here. Do not fill every NaN with zero, do not automatically delete IQR outliers, and do not overwrite the original CSV.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the project root whether Jupyter starts in the project root or notebooks/.
_SEARCH_BASES = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = None
for _base in _SEARCH_BASES:
    if (_base / "data" / "Dataset.csv").is_file():
        PROJECT_ROOT = _base
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project root. Expected data/Dataset.csv beneath the current folder. "
        f"Current directory: {Path.cwd().resolve()}"
    )

DATA_PATH = PROJECT_ROOT / "data" / "Dataset.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
REPORT_ROOT = PROJECT_ROOT / "reports"
OUTPUT_ROOT.mkdir(exist_ok=True)
REPORT_ROOT.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Original dataset:", DATA_PATH)
print("Dataset exists:", DATA_PATH.is_file())

PREP_DIR = OUTPUT_ROOT / "02_preprocessing"
PREP_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PATH = PREP_DIR / "cleaned_dataset.csv.gz"

Project root: /Users/mohammedfaisalpasha/Downloads/SepsisShield_NIErnaya_Phase2
Original dataset: /Users/mohammedfaisalpasha/Downloads/SepsisShield_NIErnaya_Phase2/data/Dataset.csv
Dataset exists: True


## 2. Load original data and define key columns

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)
original_shape = df.shape
original_missing = df.isna().sum()
print("Original shape:", original_shape)

required = ["Patient_ID", "Hour", "ICULOS", "SepsisLabel"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

ID_COLS = ["Patient_ID", "Hour", "ICULOS", "SepsisLabel"]
NON_MEASUREMENT = set(ID_COLS + ["Unnamed: 0", "Age", "Gender", "Unit1", "Unit2", "HospAdmTime"])
MEASUREMENT_COLS = [c for c in df.select_dtypes(include=np.number).columns if c not in NON_MEASUREMENT]

Original shape: (1552210, 44)


## 3. Structural audit before making changes

Check exact duplicate rows and duplicate patient-hour keys. If a patient-hour key appears more than once with different values, stop and investigate rather than arbitrarily keeping one record.

In [3]:
exact_duplicate_rows = int(df.duplicated().sum())
duplicate_patient_hour = int(df.duplicated(["Patient_ID", "Hour"]).sum())
invalid_labels = sorted(set(df["SepsisLabel"].dropna().unique()) - {0,1})
nonfinite_counts = {}
for col in df.select_dtypes(include=np.number).columns:
    nonfinite_counts[col] = int((~np.isfinite(df[col].to_numpy(dtype="float64", copy=False))).sum())
nonfinite_total = int(sum(nonfinite_counts.values()))

print("Exact duplicate rows:", exact_duplicate_rows)
print("Duplicate Patient_ID + Hour pairs:", duplicate_patient_hour)
print("Unexpected SepsisLabel values:", invalid_labels)
print("Non-finite numeric cells:", nonfinite_total)

if duplicate_patient_hour:
    raise ValueError("Duplicate patient-hour keys found. Investigate conflicting records before proceeding.")
if invalid_labels:
    raise ValueError(f"Unexpected target labels: {invalid_labels}")

Exact duplicate rows: 0
Duplicate Patient_ID + Hour pairs: 0
Unexpected SepsisLabel values: []
Non-finite numeric cells: 43512155


## 4. Remove only the verified redundant index column

In this actual CSV, `Unnamed: 0` is an exact duplicate of `Hour`. Remove it from the working dataset but retain the untouched source file in `data/`.

In [4]:
removed_columns = []
if "Unnamed: 0" in df.columns and df["Unnamed: 0"].equals(df["Hour"]):
    df = df.drop(columns=["Unnamed: 0"])
    removed_columns.append("Unnamed: 0")

# Remove exact duplicate full rows if present, and record the change.
# This file currently has zero such rows.
if exact_duplicate_rows:
    df = df.drop_duplicates().copy()

# Convert any +/- infinity in numeric measurements to missing values.
# This does not impute or otherwise invent a measurement.
for col in df.select_dtypes(include=np.number).columns:
    vals = df[col].to_numpy(dtype="float64", copy=False)
    if np.isinf(vals).any():
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

# Ensure chronological order within each patient.
df = df.sort_values(["Patient_ID", "Hour"], kind="mergesort").reset_index(drop=True)

# Verify target is integer-like binary and non-missing.
if df["SepsisLabel"].isna().any() or not set(df["SepsisLabel"].unique()).issubset({0,1}):
    raise ValueError("SepsisLabel must remain non-missing and binary.")

print("Removed verified redundant columns:", removed_columns)
print("Cleaned working shape:", df.shape)

Removed verified redundant columns: ['Unnamed: 0']
Cleaned working shape: (1552210, 43)


## 5. Outlier screening — no automatic deletion

Use the IQR rule to identify statistically unusual values in clinical measurements. An IQR flag is only a review indicator; it is not a clinical validity rule. All rows and valid extreme values are retained.

In [5]:
outlier_rows = []
for col in MEASUREMENT_COLS:
    if col not in df.columns:
        continue
    x = df[col].dropna()
    if len(x) == 0:
        continue
    q1, q3 = x.quantile([.25,.75])
    iqr = q3-q1
    low, high = q1-1.5*iqr, q3+1.5*iqr
    flag = (x < low) | (x > high)
    outlier_rows.append({"feature":col,"observed_n":len(x),"q1":q1,"q3":q3,
                         "lower_iqr_fence":low,"upper_iqr_fence":high,
                         "flagged_n":int(flag.sum()),"flagged_pct":float(flag.mean()*100),
                         "action":"flagged_for_review_not_deleted"})
outlier_report = pd.DataFrame(outlier_rows).sort_values("flagged_pct",ascending=False)
display(outlier_report.head(20))
outlier_report.to_csv(PREP_DIR/"outlier_review_report.csv",index=False)

,feature,observed_n,q1,q3,lower_iqr_fence,upper_iqr_fence,flagged_n,flagged_pct,action
27,TroponinI,14781,0.04,3.98,-5.870,9.890,2500,16.913605,flagged_for_review_not_deleted
13,SaO2,53561,94.00,98.00,88.000,104.000,8198,15.305913,flagged_for_review_not_deleted
14,AST,25183,22.00,111.00,-111.500,244.500,3811,15.133225,flagged_for_review_not_deleted
17,Calcium,91331,7.70,8.70,6.200,10.200,13435,14.710230,flagged_for_review_not_deleted
20,Bilirubin_direct,2990,0.20,1.70,-2.050,3.950,372,12.441472,flagged_for_review_not_deleted
26,Bilirubin_total,23141,0.50,1.70,-1.300,3.500,2769,11.965775,flagged_for_review_not_deleted
19,Creatinine,94616,0.70,1.43,-0.395,2.525,11139,11.772850,flagged_for_review_not_deleted
30,PTT,45699,27.80,42.80,5.300,65.300,4855,10.623865,flagged_for_review_not_deleted
10,FiO2,129365,0.40,0.60,0.100,0.900,13191,10.196730,flagged_for_review_not_deleted
16,Alkalinephos,24941,54.00,108.00,-27.000,189.000,2135,8.560202,flagged_for_review_not_deleted


## 5B. Conservative validity checks (flag only; do not delete)

IQR fences identify statistical extremes, not invalid clinical records. This additional screen flags only clear structural impossibilities or out-of-domain codes for manual review. It does not remove or replace any row.

In [6]:
validity_rules = {
    "HR": lambda s: s.le(0),
    "O2Sat": lambda s: s.lt(0) | s.gt(100),
    "Temp": lambda s: s.le(0),
    "SBP": lambda s: s.le(0),
    "MAP": lambda s: s.le(0),
    "DBP": lambda s: s.le(0),
    "Resp": lambda s: s.le(0),
    "EtCO2": lambda s: s.lt(0),
    "Age": lambda s: s.lt(0) | s.gt(120),
    "Gender": lambda s: ~s.isin([0, 1]),
    "Hour": lambda s: s.lt(0),
    "ICULOS": lambda s: s.lt(1),
    "SepsisLabel": lambda s: ~s.isin([0, 1]),
}

validity_rows = []
for col, rule in validity_rules.items():
    if col in df.columns:
        mask = df[col].notna() & rule(df[col])
        validity_rows.append({
            "feature": col,
            "flagged_count": int(mask.sum()),
            "flagged_pct_of_rows": float(mask.mean() * 100),
            "action": "flag_for_manual_review_not_deleted"
        })
validity_report = pd.DataFrame(validity_rows)
display(validity_report)
validity_report.to_csv(PREP_DIR / "conservative_validity_flags.csv", index=False)


,feature,flagged_count,flagged_pct_of_rows,action
0,HR,0,0.0,flag_for_manual_review_not_deleted
1,O2Sat,0,0.0,flag_for_manual_review_not_deleted
2,Temp,0,0.0,flag_for_manual_review_not_deleted
3,SBP,0,0.0,flag_for_manual_review_not_deleted
4,MAP,0,0.0,flag_for_manual_review_not_deleted
5,DBP,0,0.0,flag_for_manual_review_not_deleted
6,Resp,0,0.0,flag_for_manual_review_not_deleted
7,EtCO2,0,0.0,flag_for_manual_review_not_deleted
8,Age,0,0.0,flag_for_manual_review_not_deleted
9,Gender,0,0.0,flag_for_manual_review_not_deleted


## 6. Missing-value strategy

**Decision for this Phase 2 handoff:** preserve the missing values. Do not fill with zero or global medians here.

Reasons:
- The frequency of measurement is itself worth investigating.
- Forward filling across patients is invalid.
- Global imputation before patient split can leak information from held-out patients.
- Later model pipelines should fit imputers on training patients only.
- Person 3 creates explicit missingness features from the raw missingness pattern.

In [7]:
missing_before = original_missing.reindex(df.columns).fillna(0).astype(int)
missing_after = df.isna().sum()
missing_report = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after,
    "missing_pct_after": df.isna().mean()*100
}).sort_values("missing_pct_after",ascending=False)
display(missing_report.head(20))
missing_report.to_csv(PREP_DIR/"missingness_before_after.csv")

,missing_before,missing_after,missing_pct_after
Bilirubin_direct,1549220,1549220,99.807371
Fibrinogen,1541968,1541968,99.340167
TroponinI,1537429,1537429,99.047745
Bilirubin_total,1529069,1529069,98.509158
Alkalinephos,1527269,1527269,98.393194
AST,1527027,1527027,98.377604
Lactate,1510764,1510764,97.329872
PTT,1506511,1506511,97.055875
SaO2,1498649,1498649,96.549372
EtCO2,1494574,1494574,96.286843


## 7. Validate row counts, patient-hour uniqueness and time sequence

In [8]:
final_duplicate_rows = int(df.duplicated().sum())
final_duplicate_patient_hour = int(df.duplicated(["Patient_ID","Hour"]).sum())
hour_diff = df.groupby("Patient_ID",sort=False)["Hour"].diff()
nonconsecutive = int((hour_diff.notna() & hour_diff.ne(1)).sum())

assert final_duplicate_rows == 0, "Unexpected duplicate rows remain."
assert final_duplicate_patient_hour == 0, "Duplicate patient-hour combinations remain."
assert df["Patient_ID"].nunique() > 0
assert df["SepsisLabel"].isin([0,1]).all()

print("Rows retained:",len(df),"of",original_shape[0])
print("Columns after structural cleanup:",df.shape[1])
print("Patients:",df.Patient_ID.nunique())
print("Duplicate rows:",final_duplicate_rows)
print("Duplicate patient-hour pairs:",final_duplicate_patient_hour)
print("Non-consecutive Hour gaps:",nonconsecutive)
print("Missing values preserved:",int(df.isna().sum().sum()))

Rows retained: 1552210 of 1552210
Columns after structural cleanup: 43
Patients: 40336
Duplicate rows: 0
Duplicate patient-hour pairs: 0
Non-consecutive Hour gaps: 0
Missing values preserved: 43512155


## 8. Export cleaned data and handoff reports

The default portable output is compressed CSV (`.csv.gz`), so no extra Parquet engine is required. The file can be loaded with `pd.read_csv(path, compression="gzip")`.

In [9]:
df.to_csv(CLEAN_PATH,index=False,compression={"method": "gzip", "compresslevel": 1})

preprocessing_summary = {
    "input_file": str(DATA_PATH),
    "original_rows": int(original_shape[0]),
    "original_columns": int(original_shape[1]),
    "output_rows": int(df.shape[0]),
    "output_columns": int(df.shape[1]),
    "patients": int(df.Patient_ID.nunique()),
    "removed_columns": removed_columns,
    "exact_duplicate_rows_before": exact_duplicate_rows,
    "duplicate_patient_hour_before": duplicate_patient_hour,
    "nonconsecutive_hour_gaps_after": nonconsecutive,
    "missing_values_preserved": int(df.isna().sum().sum()),
    "output_file": str(CLEAN_PATH)
}
(PREP_DIR/"preprocessing_summary.json").write_text(json.dumps(preprocessing_summary,indent=2))
print("Saved cleaned dataset:",CLEAN_PATH)
print("Output size (MB):",round(CLEAN_PATH.stat().st_size/1024**2,2))

Saved cleaned dataset: /Users/mohammedfaisalpasha/Downloads/SepsisShield_NIErnaya_Phase2/outputs/02_preprocessing/cleaned_dataset.csv.gz
Output size (MB): 33.83


## 9. Person 2 handoff

Give Person 3:
- `cleaned_dataset.csv.gz`
- `outlier_review_report.csv`
- `missingness_before_after.csv`
- `preprocessing_summary.json`

**Do not claim all outliers were removed or missing values were filled.** This handoff deliberately retains valid extreme values and missing clinical observations. Any later imputation must be fitted within the training data only.